In [1]:
# NORTHSTAR — Solace Browser: Papers & Claims Validation QA
# DNA: claims_qa = pricing(consistency) x features(OAuth3+Evidence+Recipes) x papers(referenced) x legal(no_fabrication)
# Template: solace-cli/notebooks/qa-templates/template-page-qa.ipynb (adapted for file-based)
# Towers: T1(Masters) T2(Customers) T3(Competitors) T6(Hackers) T9(World)
# Auth: 65537 | Papers: 46,47,49,50
#
# Cross-page claim consistency validation — reads all HTML files and checks:
# 1. Pricing claims match across pages ($0/$8/$28/$88/$188)
# 2. Feature claims (OAuth3, Evidence Chain, Recipes, BYOK) are consistent
# 3. No fabricated testimonials or unsubstantiated claims
# 4. Legal pages exist and are linked
# REAL assertions. No mocks. No stubs.

import json, re, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "solace-browser-papers-claims-qa"
NOTEBOOK_PATH = "notebooks/qa/34-papers-claims-qa.ipynb"
PROJECT = "solace-browser"
MIN_SCORE = 70

BROWSER = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER / "web"
PAPERS = Path("/home/phuc/projects/solace-cli/papers")

# Load all HTML content for cross-page analysis
ALL_HTML = {}
for p in sorted(WEB.glob("*.html")):
    ALL_HTML[p.stem] = p.read_text(encoding="utf-8")
for p in sorted((WEB / "docs").glob("*.html")):
    ALL_HTML[f"docs/{p.stem}"] = p.read_text(encoding="utf-8")

results = []

def record(probe_id, name, passed, detail="", tower_ref=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status,
                    "detail": detail, "tower_ref": tower_ref})
    print(f"  {status}: {name}" + (f" -- {detail}" if detail and not passed else ""))

print(f"BOOTSTRAP: {len(ALL_HTML)} pages loaded, {len(list(PAPERS.glob('*.md')))} papers")
print(f"Pages: {sorted(ALL_HTML.keys())}")

BOOTSTRAP: 23 pages loaded, 61 papers
Pages: ['404', '500', 'app-detail', 'app-store', 'create-app', 'demo', 'docs', 'docs/mcp', 'docs/oauth3', 'docs/quick-start', 'download', 'glossary', 'guide', 'home', 'index', 'machine-dashboard', 'partials-footer', 'partials-header', 'schedule', 'settings', 'start', 'style-guide', 'tunnel-connect']


In [2]:
# ── P1: Pricing Consistency ──────────────────────────────────────────────────
# Tower T2(Customers) T3(Competitors): pricing claims must match across all pages
# Canonical tiers: Free($0) | Starter($8/mo) | Pro($28/mo) | Team($88/mo) | Enterprise($188/mo)

print("=== P1: Pricing Consistency ===")

CANONICAL_PRICES = {"$0": "Free", "$8": "Starter", "$28": "Pro", "$88": "Team", "$188": "Enterprise"}
PRICE_PATTERN = re.compile(r'\$(?:0|8|28|88|188)(?:/mo)?')

# Pages that mention pricing
pricing_pages = {}
for name, html in ALL_HTML.items():
    prices_found = set(PRICE_PATTERN.findall(html))
    if prices_found:
        pricing_pages[name] = prices_found

record("P1-01", "At least one page mentions pricing",
       len(pricing_pages) > 0,
       f"Found {len(pricing_pages)} pages with pricing: {sorted(pricing_pages.keys())}",
       "T2")

# Check that no page has a rogue price that doesn't match canonical
ROGUE_PRICE = re.compile(r'\$(\d+)(?:/mo)')
for name, html in ALL_HTML.items():
    rogue_matches = ROGUE_PRICE.findall(html)
    rogue = [f"${m}" for m in rogue_matches if f"${m}" not in CANONICAL_PRICES and m not in ("0", "3", "5")]
    # Exclude small amounts that are per-run costs, not tier prices
    rogue = [r for r in rogue if int(r.replace("$","")) > 5]
    record(f"P1-02-{name}", f"No rogue tier prices on {name}",
           len(rogue) == 0,
           f"Rogue prices: {rogue}" if rogue else "",
           "T2")

# Pages that show tiers should show at least Free + Starter + Pro
TIER_PAGES = ["start", "settings"]
for page in TIER_PAGES:
    if page in ALL_HTML:
        html = ALL_HTML[page]
        has_free = bool(re.search(r'free|Free|\$0', html))
        has_starter = bool(re.search(r'[Ss]tarter.*\$8|\$8.*[Ss]tarter|\$8/mo', html))
        has_pro = bool(re.search(r'[Pp]ro.*\$28|\$28.*[Pp]ro|\$28/mo', html))
        record(f"P1-03-{page}", f"{page} shows Free+Starter+Pro tiers",
               has_free and has_starter and has_pro,
               f"Free={has_free}, Starter={has_starter}, Pro={has_pro}",
               "T2")

print(f"  Pricing pages: {sorted(pricing_pages.keys())}")

=== P1: Pricing Consistency ===
  PASS: At least one page mentions pricing
  PASS: No rogue tier prices on 404
  PASS: No rogue tier prices on 500
  PASS: No rogue tier prices on app-detail
  PASS: No rogue tier prices on app-store
  PASS: No rogue tier prices on create-app
  PASS: No rogue tier prices on demo
  PASS: No rogue tier prices on docs
  PASS: No rogue tier prices on download
  PASS: No rogue tier prices on glossary
  PASS: No rogue tier prices on guide
  PASS: No rogue tier prices on home
  PASS: No rogue tier prices on index
  PASS: No rogue tier prices on machine-dashboard
  PASS: No rogue tier prices on partials-footer
  PASS: No rogue tier prices on partials-header
  PASS: No rogue tier prices on schedule
  PASS: No rogue tier prices on settings
  PASS: No rogue tier prices on start
  PASS: No rogue tier prices on style-guide
  PASS: No rogue tier prices on tunnel-connect
  PASS: No rogue tier prices on docs/mcp
  PASS: No rogue tier prices on docs/oauth3
  PASS: No rog

In [3]:
# ── P2: Feature Claims Consistency ───────────────────────────────────────────
# Tower T1(Masters) T3(Competitors): OAuth3, Evidence Chain, Recipes, BYOK
# must be referenced consistently and not fabricated

print("=== P2: Feature Claims Consistency ===")

FEATURES = {
    "OAuth3": re.compile(r'OAuth3', re.IGNORECASE),
    "Evidence": re.compile(r'evidence[\s-]*(chain|capture|seal|log|trail|driven)', re.IGNORECASE),
    "Recipes": re.compile(r'recipe', re.IGNORECASE),
    "BYOK": re.compile(r'BYOK|bring\s+your\s+own\s+key', re.IGNORECASE),
    "Preview": re.compile(r'preview.*approv|approv.*preview|LLM.*once|preview.*before', re.IGNORECASE),
}

# Check which pages reference each feature
feature_pages = {}
for feat, pat in FEATURES.items():
    pages = [name for name, html in ALL_HTML.items() if pat.search(html)]
    feature_pages[feat] = pages
    record(f"P2-01-{feat}", f"{feat} referenced on at least 1 page",
           len(pages) >= 1,
           f"Found on: {pages}",
           "T1")

# OAuth3 should have its own docs page
record("P2-02", "OAuth3 has dedicated docs page",
       "docs/oauth3" in ALL_HTML,
       "docs/oauth3.html" if "docs/oauth3" in ALL_HTML else "MISSING docs/oauth3.html",
       "T1")

# OAuth3 JS confirm dialog should exist and be loaded
oauth3_js = (WEB / "js" / "yinyang-oauth3-confirm.js")
record("P2-03", "OAuth3 confirm JS exists",
       oauth3_js.exists(),
       str(oauth3_js),
       "T6")

# Count pages that load the OAuth3 confirm script
oauth3_loaded = [name for name, html in ALL_HTML.items()
                 if "yinyang-oauth3-confirm.js" in html]
record("P2-04", "OAuth3 confirm JS loaded on 3+ pages",
       len(oauth3_loaded) >= 3,
       f"Loaded on {len(oauth3_loaded)} pages: {oauth3_loaded}",
       "T6")

# Recipes should mention replay/deterministic
recipe_replay = [name for name, html in ALL_HTML.items()
                 if re.search(r'replay|deterministic', html, re.IGNORECASE)
                 and re.search(r'recipe', html, re.IGNORECASE)]
record("P2-05", "Recipe+replay co-occur (cost inversion claim supported)",
       len(recipe_replay) >= 1,
       f"Pages: {recipe_replay}",
       "T3")

# BYOK claim: user brings own key = $0 LLM cost
byok_pages = feature_pages.get("BYOK", [])
record("P2-06", "BYOK mentioned on download or settings page",
       any(p in byok_pages for p in ["download", "settings", "start", "demo"]),
       f"BYOK pages: {byok_pages}",
       "T2")

print(f"  Feature coverage: {', '.join(f'{k}={len(v)}' for k, v in feature_pages.items())}")

=== P2: Feature Claims Consistency ===
  PASS: OAuth3 referenced on at least 1 page
  PASS: Evidence referenced on at least 1 page
  PASS: Recipes referenced on at least 1 page
  PASS: BYOK referenced on at least 1 page
  PASS: Preview referenced on at least 1 page
  PASS: OAuth3 has dedicated docs page
  PASS: OAuth3 confirm JS exists
  PASS: OAuth3 confirm JS loaded on 3+ pages
  PASS: Recipe+replay co-occur (cost inversion claim supported)
  PASS: BYOK mentioned on download or settings page
  Feature coverage: OAuth3=20, Evidence=10, Recipes=10, BYOK=3, Preview=5


In [4]:
# ── P3: No Fabricated Testimonials ───────────────────────────────────────────
# Tower T6(Hackers) T1(Masters): absolutely no fake testimonials, quotes, or reviews
# Check for common fabrication patterns

print("=== P3: No Fabricated Testimonials ===")

TESTIMONIAL_PATTERNS = [
    re.compile(r'<blockquote[^>]*class="[^"]*testimonial', re.IGNORECASE),
    re.compile(r'class="[^"]*testimonial[^"]*"', re.IGNORECASE),
    re.compile(r'class="[^"]*customer-quote[^"]*"', re.IGNORECASE),
    re.compile(r'class="[^"]*review-card[^"]*"', re.IGNORECASE),
]

ATTRIBUTION_PATTERNS = [
    re.compile(r'(?:—|&mdash;)\s*([A-Z][a-z]+\s+[A-Z][a-z]+)', re.MULTILINE),
]

# Known real people/entities used as design references or UI labels
KNOWN_REAL = {"Jony Ive", "Steve Jobs", "Seth Godin", "Russell Brunson",
              "Don Norman", "Dieter Rams", "Dragon Rider", "Phuc Nguyen",
              "Apple Apps", "Apple App", "Gmail Inbox", "Solace Browser",
              "Agent Inspector", "Agent Instructions",
              "Business Automation", "Awaiting Approval", "The Yinyang",
              "The Yin", "No Apps"}

for name, html in ALL_HTML.items():
    has_testimonial = any(p.search(html) for p in TESTIMONIAL_PATTERNS)
    record(f"P3-01-{name}", f"No fabricated testimonial elements on {name}",
           not has_testimonial,
           "Testimonial element found — verify it's real, not fabricated" if has_testimonial else "",
           "T6")

# Check for attributed quotes that might be fake
attributed_quotes = {}
for name, html in ALL_HTML.items():
    for pat in ATTRIBUTION_PATTERNS:
        matches = pat.findall(html)
        if matches:
            suspicious = [m for m in matches if m not in KNOWN_REAL]
            if suspicious:
                attributed_quotes[name] = suspicious

record("P3-02", "No suspicious attributed quotes across site",
       len(attributed_quotes) == 0,
       f"Suspicious on: {attributed_quotes}" if attributed_quotes else "All quotes from known sources",
       "T6")

# No star ratings or review scores (exclude style-guide which is a design showcase)
STAR_PATTERNS = re.compile(r'★{3,5}|⭐|class="[^"]*star-rating', re.IGNORECASE)
for name, html in ALL_HTML.items():
    if name == "style-guide":
        continue  # style-guide is a design component showcase, not content
    has_stars = bool(STAR_PATTERNS.search(html))
    record(f"P3-03-{name}", f"No fake star ratings on {name}",
           not has_stars,
           "Star rating found — must be backed by real data" if has_stars else "",
           "T6")

print(f"  Attributed quotes found: {len(attributed_quotes)} pages")

=== P3: No Fabricated Testimonials ===
  PASS: No fabricated testimonial elements on 404
  PASS: No fabricated testimonial elements on 500
  PASS: No fabricated testimonial elements on app-detail
  PASS: No fabricated testimonial elements on app-store
  PASS: No fabricated testimonial elements on create-app
  PASS: No fabricated testimonial elements on demo
  PASS: No fabricated testimonial elements on docs
  PASS: No fabricated testimonial elements on download
  PASS: No fabricated testimonial elements on glossary
  PASS: No fabricated testimonial elements on guide
  PASS: No fabricated testimonial elements on home
  PASS: No fabricated testimonial elements on index
  PASS: No fabricated testimonial elements on machine-dashboard
  PASS: No fabricated testimonial elements on partials-footer
  PASS: No fabricated testimonial elements on partials-header
  PASS: No fabricated testimonial elements on schedule
  PASS: No fabricated testimonial elements on settings
  PASS: No fabricated test

In [5]:
# ── P4: Legal Pages Exist and Are Linked ─────────────────────────────────────
# Tower T6(Hackers) T9(World): privacy, terms, security.txt, glossary

print("=== P4: Legal Pages & Compliance ===")

# Check that essential legal/compliance pages exist
REQUIRED_PAGES = {
    "glossary": "glossary.html",
    "404": "404.html",
    "500": "500.html",
}

REQUIRED_DOCS = {
    "docs/oauth3": "docs/oauth3.html",
    "docs/mcp": "docs/mcp.html",
    "docs/quick-start": "docs/quick-start.html",
}

for name, filename in {**REQUIRED_PAGES, **REQUIRED_DOCS}.items():
    exists = name in ALL_HTML
    record(f"P4-01-{name}", f"{filename} exists",
           exists, "" if exists else f"MISSING: {filename}", "T6")

# Footer should link to privacy/terms/glossary
footer_path = WEB / "partials-footer.html"
if footer_path.exists():
    footer = footer_path.read_text(encoding="utf-8")
    for link_text, href_pat in [
        ("privacy", r'href="[^"]*privacy'),
        ("terms", r'href="[^"]*terms'),
        ("glossary", r'href="[^"]*glossary'),
        ("security", r'href="[^"]*security'),
    ]:
        found = bool(re.search(href_pat, footer, re.IGNORECASE))
        record(f"P4-02-{link_text}", f"Footer links to {link_text}",
               found, "" if found else f"Missing {link_text} link in footer", "T6")

# Header should link to key pages
header_path = WEB / "partials-header.html"
if header_path.exists():
    header = header_path.read_text(encoding="utf-8")
    for link_text, href_pat in [
        ("oauth3", r'href="[^"]*oauth3'),
        ("guide", r'href="[^"]*guide'),
        ("app-store", r'href="[^"]*app-store'),
        ("download", r'href="[^"]*download'),
    ]:
        found = bool(re.search(href_pat, header, re.IGNORECASE))
        record(f"P4-03-{link_text}", f"Header links to {link_text}",
               found, "" if found else f"Missing {link_text} link in header", "T9")

# security.txt should exist in web root or .well-known
security_txt = WEB / "security.txt"
wellknown_security = WEB / ".well-known" / "security.txt"
record("P4-04", "security.txt exists",
       security_txt.exists() or wellknown_security.exists(),
       str(security_txt) if security_txt.exists() else str(wellknown_security) if wellknown_security.exists() else "MISSING",
       "T6")

# robots.txt and sitemap.xml
robots = WEB / "robots.txt"
sitemap = WEB / "sitemap.xml"
record("P4-05", "robots.txt exists", robots.exists(), "", "T9")
record("P4-06", "sitemap.xml exists", sitemap.exists(), "", "T9")

=== P4: Legal Pages & Compliance ===
  PASS: glossary.html exists
  PASS: 404.html exists
  PASS: 500.html exists
  PASS: docs/oauth3.html exists
  PASS: docs/mcp.html exists
  PASS: docs/quick-start.html exists
  PASS: Footer links to privacy
  PASS: Footer links to terms
  PASS: Footer links to glossary
  PASS: Footer links to security
  PASS: Header links to oauth3
  PASS: Header links to guide
  PASS: Header links to app-store
  PASS: Header links to download
  PASS: security.txt exists
  PASS: robots.txt exists
  PASS: sitemap.xml exists


In [6]:
# ── P5: Paper References & Glossary Accuracy ────────────────────────────────
# Tower T1(Masters): paper numbers referenced in HTML should exist in papers/

print("=== P5: Paper References & Glossary Accuracy ===")

# Find all paper references in HTML (e.g., "Paper 06", "Paper 09", "paper-09")
PAPER_REF = re.compile(r'[Pp]aper\s*(?:#?\s*)?(\d{1,3})')
referenced_papers = set()
for name, html in ALL_HTML.items():
    matches = PAPER_REF.findall(html)
    for m in matches:
        referenced_papers.add(int(m))

# Check that each referenced paper actually exists
existing_papers = set()
for p in PAPERS.glob("*.md"):
    match = re.match(r'(\d+)-', p.name)
    if match:
        existing_papers.add(int(match.group(1)))

for paper_num in sorted(referenced_papers):
    padded = f"{paper_num:02d}"
    record(f"P5-01-{padded}", f"Referenced Paper {padded} exists in papers/",
           paper_num in existing_papers,
           f"papers/{padded}-*.md" if paper_num in existing_papers else f"MISSING paper {padded}",
           "T1")

record("P5-02", "All referenced papers exist",
       referenced_papers.issubset(existing_papers),
       f"Referenced: {sorted(referenced_papers)}, Missing: {sorted(referenced_papers - existing_papers)}",
       "T1")

# Glossary accuracy: key terms should have definitions
if "glossary" in ALL_HTML:
    glossary = ALL_HTML["glossary"]
    GLOSSARY_TERMS = ["OAuth3", "Recipe", "Evidence", "BYOK", "Preview",
                      "YinYang", "Twin", "Budget", "Scope", "Rung"]
    for term in GLOSSARY_TERMS:
        found = bool(re.search(re.escape(term), glossary, re.IGNORECASE))
        record(f"P5-03-{term}", f"Glossary defines '{term}'",
               found, "" if found else f"'{term}' missing from glossary", "T1")

print(f"  Referenced papers: {sorted(referenced_papers)}")
print(f"  Existing papers: {len(existing_papers)}")

=== P5: Paper References & Glossary Accuracy ===
  PASS: Referenced Paper 04 exists in papers/
  PASS: Referenced Paper 06 exists in papers/
  PASS: Referenced Paper 07 exists in papers/
  PASS: Referenced Paper 08 exists in papers/
  PASS: Referenced Paper 09 exists in papers/
  PASS: Referenced Paper 24 exists in papers/
  PASS: Referenced Paper 40 exists in papers/
  PASS: All referenced papers exist
  PASS: Glossary defines 'OAuth3'
  PASS: Glossary defines 'Recipe'
  PASS: Glossary defines 'Evidence'
  PASS: Glossary defines 'BYOK'
  PASS: Glossary defines 'Preview'
  PASS: Glossary defines 'YinYang'
  PASS: Glossary defines 'Twin'
  PASS: Glossary defines 'Budget'
  PASS: Glossary defines 'Scope'
  PASS: Glossary defines 'Rung'
  Referenced papers: [4, 6, 7, 8, 9, 24, 40]
  Existing papers: 55


In [7]:
# ── EVIDENCE SUMMARY ─────────────────────────────────────────────────────────
total = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
failed = sum(1 for r in results if r["status"] == "FAIL")
score = (passed / total * 100) if total else 0

print(f"\n{'='*60}")
print(f"EVIDENCE SUMMARY — {NORTHSTAR}")
print(f"{'='*60}")
print(f"Total probes: {total}")
print(f"Passed:       {passed}")
print(f"Failed:       {failed}")
print(f"Score:        {score:.1f}%")
print(f"MIN_SCORE:    {MIN_SCORE}%")
print(f"Converged:    {'YES' if score >= MIN_SCORE else 'NO'}")

if failed > 0:
    print(f"\nFAILURES:")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  FAIL: {r['name']} -- {r['detail']}")

evidence = {
    "notebook": NOTEBOOK_PATH,
    "northstar": NORTHSTAR,
    "project": PROJECT,
    "total": total,
    "passed": passed,
    "failed": failed,
    "score": round(score, 1),
    "min_score": MIN_SCORE,
    "converged": score >= MIN_SCORE,
    "timestamp": datetime.now().isoformat(),
    "hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16],
}
print(f"\nEvidence hash: {evidence['hash']}")
print(json.dumps(evidence, indent=2))


EVIDENCE SUMMARY — solace-browser-papers-claims-qa
Total probes: 117
Passed:       117
Failed:       0
Score:        100.0%
MIN_SCORE:    70%
Converged:    YES

Evidence hash: 9944c904157e7d52
{
  "notebook": "notebooks/qa/34-papers-claims-qa.ipynb",
  "northstar": "solace-browser-papers-claims-qa",
  "project": "solace-browser",
  "total": 117,
  "passed": 117,
  "failed": 0,
  "score": 100.0,
  "min_score": 70,
  "converged": true,
  "timestamp": "2026-03-06T10:27:27.077344",
  "hash": "9944c904157e7d52"
}
